In [ ]:
# This setup is only needed in Colab, which opens the notebook without cloning
# the repo it came from. If you already have the repo, skip this cell.
!git clone https://github.com/yiftachbeer/notebook-to-production/ -b act2-extract
%cd notebook-to-production
import sys
sys.path.insert(0, ".")

# Depth + Segmentation -> Colored 3D Point Cloud

Run a monocular depth model and a semantic segmentation model on a driving
image, then back-project every pixel into a 3D point cloud colored by its
semantic class, using the camera's real calibration.

The pipeline now lives in small modules (`data.py`, `depth.py`, `seg.py`,
`geometry.py`, `viz.py`). This notebook is the single-frame demo; `run.py`
runs the same pipeline over the whole dataset and logs metrics.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from data import StereoDataset
from models import HfDepthModel, HfSegmentationModel
from geometry import backproject
from viz import show_image, show_depth, show_seg, show_cloud

## Load the data

In [ ]:
!wget https://stereo-vision.s3.eu-west-3.amazonaws.com/stereo_data.zip && jar xf stereo_data.zip

In [ ]:
frames = StereoDataset("stereo_data")
frame = frames["000009"]
show_image(frame["image"])

## Step 1 - Depth

In [ ]:
depth_model = HfDepthModel()
depth_m = depth_model.predict(frame)
show_depth(depth_m)

## Step 2 - Segmentation

In [ ]:
segmentation_model = HfSegmentationModel()
seg_color = segmentation_model.predict(frame)
show_seg(seg_color)

## Step 3 - 3D point cloud

In [ ]:
points, colors = backproject(depth_m, seg_color, frame["P2"])
show_cloud(points, colors)

## The whole dataset

One frame looks great, but we can't eyeball every frame. `run.py` runs the
pipeline over the full dataset and reduces each frame to a few metrics
(written to `metrics.csv`).

In [ ]:
!python run.py